# Sharding & device meshes — visual learning path

Progressive examples. **All simulation cells run on CPU** (no GPU needed).
The last section is copy-paste code for a real multi-GPU box.

| Step | Idea |
|------|------|
| 0 | What is a device mesh? |
| 1 | Shard vs replicate |
| 2 | 1D mesh — row / column shard |
| 3 | 2D mesh — shard both axes |
| 4 | Named mesh dims (DP / TP style) |
| 5 | Matmul under sharding (why layout matters) |
| 6 | Real PyTorch `DeviceMesh` + `DTensor` (multi-GPU) |

## 0. Setup — tiny simulator

We pretend ranks are GPUs. A mesh is just a grid of ranks.

In [ ]:
from __future__ import annotations

import numpy as np
from dataclasses import dataclass
from typing import Sequence


@dataclass(frozen=True)
class Mesh:
    """Logical device mesh. shape=(2,4) → 8 ranks labeled by (i,j)."""
    shape: tuple[int, ...]
    names: tuple[str, ...] | None = None

    def __post_init__(self):
        if self.names is not None and len(self.names) != len(self.shape):
            raise ValueError("names must match mesh ndim")

    @property
    def ndim(self) -> int:
        return len(self.shape)

    @property
    def size(self) -> int:
        return int(np.prod(self.shape))

    def coords(self):
        """All (coord_tuple, global_rank) in the mesh."""
        for flat in range(self.size):
            coord = np.unravel_index(flat, self.shape)
            yield coord, flat

    def show(self):
        print(f"Mesh shape={self.shape}  names={self.names}  world_size={self.size}")
        if self.ndim == 1:
            row = " | ".join(f"r{r}" for r in range(self.shape[0]))
            print(f"  [{row}]")
        elif self.ndim == 2:
            H, W = self.shape
            name0 = (self.names or ("dim0", "dim1"))[0]
            name1 = (self.names or ("dim0", "dim1"))[1]
            print(f"         {name1} →")
            for i in range(H):
                cells = []
                for j in range(W):
                    rank = i * W + j
                    cells.append(f"({i},{j})=r{rank}")
                prefix = f"{name0}↓ " if i == 0 else "     "
                print(prefix + " | ".join(cells))
        else:
            for coord, rank in self.coords():
                print(f"  coord={coord} → rank {rank}")


# --- placements ---
# None  → not mapped (local-only mental model; we use Shard / Replicate only)
Shard = lambda dim: ("S", dim)      # shard tensor axis `dim` along this mesh axis
Replicate = lambda: ("R", None)     # full copy on every device along this mesh axis


def _shard_slices(length: int, n: int) -> list[slice]:
    """Even contiguous shards (last shard may be smaller if not divisible)."""
    base, rem = divmod(length, n)
    out, start = [], 0
    for i in range(n):
        size = base + (1 if i < rem else 0)
        out.append(slice(start, start + size))
        start += size
    return out


def distribute(tensor: np.ndarray, mesh: Mesh, placements: Sequence):
    """
    placements[i] describes how tensor is laid out along mesh axis i.
      Shard(k)     → split tensor dim k across mesh axis i
      Replicate()  → every rank on mesh axis i gets the full slice

    Returns dict: rank → local ndarray
    """
    if len(placements) != mesh.ndim:
        raise ValueError("one placement per mesh dim")

    # For each tensor dim, collect how many ways it is sharded (product of mesh sizes)
    # Build per-rank indexers.
    locals_: dict[int, np.ndarray] = {}

    # Precompute shard slices for each (mesh_axis → tensor_dim) Shard placement
    mesh_axis_slices: list[list[slice] | None] = [None] * mesh.ndim
    for mesh_ax, plc in enumerate(placements):
        kind, tdim = plc
        if kind == "S":
            mesh_axis_slices[mesh_ax] = _shard_slices(tensor.shape[tdim], mesh.shape[mesh_ax])

    for coord, rank in mesh.coords():
        index = [slice(None)] * tensor.ndim
        for mesh_ax, plc in enumerate(placements):
            kind, tdim = plc
            if kind == "S":
                index[tdim] = mesh_axis_slices[mesh_ax][coord[mesh_ax]]
            # Replicate: leave slice(None) on that mesh axis contribution
        locals_[rank] = np.asarray(tensor[tuple(index)])
    return locals_


def show_shards(locals_: dict[int, np.ndarray], mesh: Mesh, title: str = ""):
    if title:
        print(title)
    print(f"{'rank':>6}  {'coord':>10}  local_shape  local_values")
    print("-" * 70)
    for coord, rank in mesh.coords():
        arr = locals_[rank]
        flat = np.array2string(arr, precision=2, suppress_small=True, max_line_width=40)
        # single-line-ish
        flat = " ".join(flat.split())
        if len(flat) > 40:
            flat = flat[:37] + "..."
        print(f"{rank:>6}  {str(coord):>10}  {str(arr.shape):>11}  {flat}")


def show_mesh_grid(locals_: dict[int, np.ndarray], mesh: Mesh, fmt=lambda a: str(a.tolist())):
    """2D mesh pretty print: each cell shows its local shard."""
    if mesh.ndim != 2:
        show_shards(locals_, mesh)
        return
    H, W = mesh.shape
    print()
    for i in range(H):
        cells = []
        for j in range(W):
            rank = i * W + j
            cells.append(f"r{rank}:{fmt(locals_[rank])}")
        print("  " + "  |  ".join(cells))
    print()


print("simulator ready")

## 1. What is a device mesh?

A **device mesh** is an N-D grid of process ranks (GPUs).

- 1D mesh `(4,)` → a line of 4 GPUs  
- 2D mesh `(2, 4)` → 2 rows × 4 cols = 8 GPUs  

Named dims (`"dp"`, `"tp"`) let you say *"shard batch along dp, shard hidden along tp"*.

In [ ]:
print("=== 1D mesh ===")
m1 = Mesh(shape=(4,), names=("gpus",))
m1.show()

print("\n=== 2D mesh (2×4) ===")
m2 = Mesh(shape=(2, 4), names=("X", "Y"))
m2.show()

print("\n=== Named like training: DP × TP ===")
m_train = Mesh(shape=(2, 4), names=("dp", "tp"))
m_train.show()

## 2. Shard vs replicate

Take a tiny global tensor and place it on a 1D mesh of 4 ranks.

| Placement | Meaning |
|-----------|---------|
| `Shard(0)` | split along tensor dim 0; each rank keeps 1/N of the rows |
| `Replicate()` | every rank gets the **full** tensor |

In [ ]:
# Global tensor: 4 rows × 3 cols, easy to read by eye
G = np.arange(12).reshape(4, 3)
print("GLOBAL tensor:\n", G)

mesh = Mesh(shape=(4,), names=("gpus",))

print("\n--- Shard(0): split ROWS across 4 GPUs ---")
row_shards = distribute(G, mesh, placements=[Shard(0)])
show_shards(row_shards, mesh)

print("\n--- Shard(1): split COLUMNS across 4 GPUs ---")
# 3 cols / 4 GPUs → uneven; still instructive
col_shards = distribute(G, mesh, placements=[Shard(1)])
show_shards(col_shards, mesh)

print("\n--- Replicate(): every GPU has the FULL tensor ---")
reps = distribute(G, mesh, placements=[Replicate()])
show_shards(reps, mesh)

### Visual check — row shard on 4 GPUs

```
GLOBAL [4×3]:
  [[ 0  1  2]
   [ 3  4  5]
   [ 6  7  8]
   [ 9 10 11]]

Shard(0) on mesh(4):
  r0 → [[0 1 2]]          # row 0
  r1 → [[3 4 5]]          # row 1
  r2 → [[6 7 8]]          # row 2
  r3 → [[9 10 11]]        # row 3
```

Local shape = `global_shape[shard_dim] / mesh_size` (along that dim).

## 3. Continuous example A — grow the tensor, same mesh

Keep mesh fixed at 4 GPUs. Watch local shapes change as we grow dim-0.

In [ ]:
mesh4 = Mesh(shape=(4,), names=("gpus",))

for rows in (4, 8, 16, 32):
    T = np.arange(rows * 2).reshape(rows, 2)
    shards = distribute(T, mesh4, placements=[Shard(0)])
    local_rows = shards[0].shape[0]
    print(f"global={T.shape}  Shard(0) on {mesh4.size} GPUs  → local≈({local_rows}, 2)  "
          f"check: {rows}/{mesh4.size}={rows/mesh4.size}")

## 4. Continuous example B — 2D mesh, one placement at a time

Mesh `(2, 4)` named `(X, Y)`. Global tensor `8×8` of readable integers.

We try **four layouts** on the **same** tensor + mesh:

1. `[Replicate, Replicate]` — full copy everywhere  
2. `[Shard(0), Replicate]` — split rows on X, copy on Y  
3. `[Replicate, Shard(1)]` — copy on X, split cols on Y  
4. `[Shard(0), Shard(1)]` — 2D shard (block layout)

In [ ]:
mesh = Mesh(shape=(2, 4), names=("X", "Y"))
mesh.show()

G = np.arange(64).reshape(8, 8)
print("\nGLOBAL 8×8:\n", G)

layouts = {
    "RR  [Replicate, Replicate]": [Replicate(), Replicate()],
    "SR  [Shard(0), Replicate] ": [Shard(0), Replicate()],
    "RS  [Replicate, Shard(1)] ": [Replicate(), Shard(1)],
    "SS  [Shard(0), Shard(1)]  ": [Shard(0), Shard(1)],
}

for name, plc in layouts.items():
    locs = distribute(G, mesh, placements=plc)
    # summarize one rank per "interesting" corner
    print(f"\n=== {name} ===")
    print(f"  local shapes: " + ", ".join(f"r{r}={locs[r].shape}" for r in range(mesh.size)))
    print("  grid of local shards (values):")
    show_mesh_grid(locs, mesh, fmt=lambda a: f"{a.shape} min={a.min()} max={a.max()}")

### What each layout means on `(2,4)`

```
Mesh:
         Y →
X↓  (0,0)=r0 | (0,1)=r1 | (0,2)=r2 | (0,3)=r3
    (1,0)=r4 | (1,1)=r5 | (1,2)=r6 | (1,3)=r7
```

| Layout | Local shape (8×8 global) | Intuition |
|--------|--------------------------|-----------|
| RR | every rank `8×8` | full replica |
| SR | every rank `4×8` | X splits rows in half; Y copies |
| RS | every rank `8×2` | Y splits cols into 4; X copies |
| SS | every rank `4×2` | block: rows÷2 and cols÷4 |

In [ ]:
# Zoom into SS block layout — print the actual numbers on each rank
locs_ss = distribute(G, mesh, placements=[Shard(0), Shard(1)])
print("SS block layout — exact local matrices:\n")
H, W = mesh.shape
for i in range(H):
    for j in range(W):
        rank = i * W + j
        print(f"rank {rank} coord=({i},{j}) local:\n{locs_ss[rank]}\n")

## 5. Continuous example C — same shards, growing mesh

Fix global `8×8`. Change mesh width. Watch local col width shrink.

In [ ]:
G = np.arange(64).reshape(8, 8)

for y in (1, 2, 4, 8):
    m = Mesh(shape=(1, y), names=("X", "Y"))
    locs = distribute(G, m, placements=[Replicate(), Shard(1)])
    print(f"mesh=(1,{y})  RS → local_shape={locs[0].shape}  "
          f"(expect cols={8 // y})")

## 6. Continuous example D — DP × TP mental model

Classic LLM training mesh:

- mesh dim `dp` → data parallel (usually shard **batch**)  
- mesh dim `tp` → tensor parallel (usually shard **hidden / vocab**)

Activation `X`: shape `[B, S, H]` — we flatten to 2D `[B, H]` for clarity.

In [ ]:
# B=4 batch, H=8 hidden  →  "activation" matrix
X = np.arange(4 * 8).reshape(4, 8)
print("Activation X [B,H]:\n", X)

mesh = Mesh(shape=(2, 4), names=("dp", "tp"))
mesh.show()

# Typical: Shard batch on dp, Shard hidden on tp
locs = distribute(X, mesh, placements=[Shard(0), Shard(1)])
print("\nplacements = [Shard(batch), Shard(hidden)]  → local [B/dp, H/tp]")
show_mesh_grid(locs, mesh, fmt=lambda a: f"{a.tolist()}")

# Weight W: [H, H_out] often sharded on columns for ColumnParallel
W = np.arange(8 * 8).reshape(8, 8) + 100
print("\nWeight W [H, H_out]:\n", W)

# Replicate on dp (same weights per DP replica), Shard cols on tp
W_locs = distribute(W, mesh, placements=[Replicate(), Shard(1)])
print("\nW placements = [Replicate(dp), Shard(1) on tp]  → ColumnParallel style")
show_mesh_grid(W_locs, mesh, fmt=lambda a: f"shape={a.shape} cols={a[0].tolist()}")

## 7. Continuous example E — why layout matters for matmul

Local matmul only works when shard axes **line up**.

- `X` sharded on cols (`Shard(1)`) and `W` sharded on rows (`Shard(0)`)  
  → each rank can do `X_local @ W_local`, then **all-reduce / all-gather** to finish.

We show the *local* products; communication is the missing piece in real systems.

In [ ]:
# 1D mesh of 4 GPUs — tensor parallel along the contraction
mesh = Mesh(shape=(4,), names=("tp",))

X = np.arange(2 * 8).reshape(2, 8).astype(float)       # [B, H]
W = np.arange(8 * 4).reshape(8, 4).astype(float)       # [H, H_out]
Y_ref = X @ W
print("X:\n", X)
print("W:\n", W)
print("Y_ref = X@W:\n", Y_ref)

# Column-parallel style: shard X on H, shard W on rows (same H split)
X_locs = distribute(X, mesh, placements=[Shard(1)])   # each rank: [2, 2]
W_locs = distribute(W, mesh, placements=[Shard(0)])   # each rank: [2, 4]

print("\n--- local matmuls (partial results) ---")
partials = []
for r in range(mesh.size):
    partial = X_locs[r] @ W_locs[r]   # [2, 4]
    partials.append(partial)
    print(f"rank {r}: X{X_locs[r].shape} @ W{W_locs[r].shape} =\n{partial}\n")

# Sum partials == full matmul  (this SUM is an AllReduce in real TP)
Y_sum = sum(partials)
print("sum(partials) == Y_ref ?", np.allclose(Y_sum, Y_ref))
print(Y_sum)

### Takeaway

```
X[:, h0] @ W[h0, :]   +   X[:, h1] @ W[h1, :]   +   ...   =   X @ W
   ↑ local on r0              ↑ local on r1

Communication = AllReduce(sum) across the TP mesh dim
```

Device mesh tells you **which ranks** participate in that AllReduce: everyone along the `tp` axis.

## 8. Continuous example F — visualize which ranks share a mesh dim

On a 2D mesh, collectives are *per group* along one axis.

In [ ]:
mesh = Mesh(shape=(2, 4), names=("dp", "tp"))
mesh.show()

print("\nGroups along tp (same dp row) — tensor-parallel AllReduce groups:")
H, W = mesh.shape
for i in range(H):
    ranks = [i * W + j for j in range(W)]
    print(f"  dp={i}: ranks {ranks}")

print("\nGroups along dp (same tp col) — data-parallel AllReduce groups:")
for j in range(W):
    ranks = [i * W + j for i in range(H)]
    print(f"  tp={j}: ranks {ranks}")

## 9. Real multi-GPU — copy to a machine with ≥2 CUDA devices

Requires: `torch` with CUDA, launched via `torchrun`.

Save as `dtensor_mesh_demo.py` and run:

```bash
# 2 GPUs, 1D mesh
torchrun --nproc_per_node=2 dtensor_mesh_demo.py --mode 1d

# 8 GPUs, 2D mesh 2×4
torchrun --nproc_per_node=8 dtensor_mesh_demo.py --mode 2d
```

In [ ]:
# %% writefile dtensor_mesh_demo.py
# Copy this entire cell into dtensor_mesh_demo.py on your GPU box.

"""
Visual DTensor + DeviceMesh demo.
Run with torchrun (see markdown above).
"""
from __future__ import annotations

import argparse
import os

import torch
import torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.tensor import DTensor, Replicate, Shard, distribute_tensor


def setup():
    dist.init_process_group("nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank, dist.get_rank(), dist.get_world_size()


def banner(rank: int, msg: str):
    dist.barrier()
    if rank == 0:
        print("\n" + "=" * 60)
        print(msg)
        print("=" * 60)
    dist.barrier()


def show_local(rank: int, name: str, dt: DTensor):
    local = dt.to_local()
    # print one rank at a time for clean logs
    for r in range(dist.get_world_size()):
        dist.barrier()
        if rank == r:
            print(
                f"[rank {rank}] {name}: global={tuple(dt.shape)} "
                f"local={tuple(local.shape)} placements={dt.placements}\n"
                f"         local_tensor=\n{local.detach().cpu()}\n"
            )
    dist.barrier()


def demo_1d(rank: int, world: int):
    mesh = init_device_mesh("cuda", (world,), mesh_dim_names=("tp",))
    banner(rank, f"1D mesh shape=({world},) names=('tp',)")

    # Each rank starts with the SAME global values via distribute_tensor from rank0 seed
    # Easier: build a global-like local then distribute from a full tensor on each rank
    # (distribute_tensor expects identical full tensor on every rank when sharding)
    full = torch.arange(8 * 4, device="cuda", dtype=torch.float32).reshape(8, 4)

    banner(rank, "Shard(0) — row parallel")
    dt = distribute_tensor(full, mesh, placements=[Shard(0)])
    show_local(rank, "Shard(0)", dt)

    banner(rank, "Shard(1) — col parallel")
    dt = distribute_tensor(full, mesh, placements=[Shard(1)])
    show_local(rank, "Shard(1)", dt)

    banner(rank, "Replicate — full copy")
    dt = distribute_tensor(full, mesh, placements=[Replicate()])
    show_local(rank, "Replicate", dt)

def demo_2d(rank: int, world: int):
    assert world == 8, "2d demo expects 8 GPUs (2×4)"
    mesh = init_device_mesh("cuda", (2, 4), mesh_dim_names=("dp", "tp"))
    banner(rank, "2D mesh (2,4) names=('dp','tp')")

    if rank == 0:
        print(mesh)
        print("mesh shape:", tuple(mesh.shape))

    full = torch.arange(8 * 8, device="cuda", dtype=torch.float32).reshape(8, 8)

    for name, plc in [
        ("RR", [Replicate(), Replicate()]),
        ("SR", [Shard(0), Replicate()]),
        ("RS", [Replicate(), Shard(1)]),
        ("SS", [Shard(0), Shard(1)]),
    ]:
        banner(rank, f"placements {name}: {plc}")
        dt = distribute_tensor(full, mesh, placements=plc)
        show_local(rank, name, dt)

    # Tiny TP matmul check on tp dim only: use a 1D submesh if available,
    # else illustrate local shapes for ColumnParallel-style weights
    banner(rank, "DP×TP activation + ColumnParallel weight shapes")
    act = torch.arange(4 * 8, device="cuda", dtype=torch.float32).reshape(4, 8)
    w = torch.arange(8 * 8, device="cuda", dtype=torch.float32).reshape(8, 8)

    act_dt = distribute_tensor(act, mesh, placements=[Shard(0), Shard(1)])  # shard B on dp, H on tp
    w_dt = distribute_tensor(w, mesh, placements=[Replicate(), Shard(1)])   # replicate on dp, shard out on tp
    show_local(rank, "activation [Shard(B), Shard(H)]", act_dt)
    show_local(rank, "weight [Replicate, Shard(out)]", w_dt)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--mode", choices=("1d", "2d"), default="1d")
    args = parser.parse_args()

    local_rank, rank, world = setup()
    if rank == 0:
        print(f"world_size={world} backend={dist.get_backend()}")

    if args.mode == "1d":
        demo_1d(rank, world)
    else:
        demo_2d(rank, world)

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

## 10. Cheatsheet

| Concept | Meaning |
|---------|---------|
| Device mesh | N-D grid of ranks; named dims = parallel strategies |
| `Shard(k)` | split tensor dim `k` across that mesh axis |
| `Replicate` | full copy along that mesh axis |
| Local shape | `global[dim] / mesh_size` for each sharded dim |
| 2D `SS` | block / grid shard |
| TP groups | ranks that share the same DP coordinate |
| DP groups | ranks that share the same TP coordinate |
| Matmul + shard on contract dim | local matmul → AllReduce sum |

**Suggested run order:** cells 0 → 8 on any laptop, then cell 9 on a GPU box with `torchrun`.